# Day 6: Advanced Analytics
## Mutual Fund Risk, Investor Behavior & Portfolio Concentration

This notebook extends the Day 4 performance analytics with:
1. Historical VaR and CVaR
2. Rolling 90-Day Sharpe Ratio
3. Investor Cohort Analysis
4. SIP Continuity Analysis
5. Sector HHI Concentration
6. Advanced Insights

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

TRADING_DAYS = 252
DATA_DIR = Path("data/processed")
CHART_DIR = Path("reports/charts")
CHART_DIR.mkdir(parents=True, exist_ok=True)

KEY_FUNDS = {
    119551: "SBI Bluechip",
    120503: "ICICI Bluechip",
    118632: "Nippon Large Cap",
    119092: "Axis Bluechip",
    120841: "Kotak Bluechip",
}

In [ ]:
nav_df = pd.read_csv(DATA_DIR / "02_nav_history.csv", parse_dates=["date"])
fund_df = pd.read_csv(DATA_DIR / "01_fund_master.csv")
txn_df = pd.read_csv(DATA_DIR / "08_investor_transactions.csv", parse_dates=["transaction_date"])
holdings_df = pd.read_csv(DATA_DIR / "09_portfolio_holdings.csv")

nav_df = nav_df.sort_values(["amfi_code", "date"])
nav_df["daily_return"] = nav_df.groupby("amfi_code")["nav"].transform(
    lambda x: (x / x.shift(1)) - 1
)

print(f"NAV records: {len(nav_df):,} | Funds: {nav_df['amfi_code'].nunique()}")
print(f"Transactions: {len(txn_df):,} | Holdings: {len(holdings_df):,}")

## Task 1: Historical VaR and CVaR

**Value at Risk (VaR)** estimates the maximum expected daily loss at a given confidence level. We use the **Historical VaR (95%)** — the 5th percentile of daily returns.

**CVaR (Expected Shortfall)** is the average return on days worse than the VaR threshold, giving a sense of tail-risk beyond VaR.

In [ ]:
var_rows = []
for code, grp in nav_df.groupby("amfi_code"):
    rets = grp["daily_return"].dropna()
    var_95 = rets.quantile(0.05)
    cvar_95 = rets[rets <= var_95].mean()
    name = fund_df.loc[fund_df["amfi_code"] == code, "scheme_name"].iloc[0]
    var_rows.append({
        "amfi_code": code,
        "scheme_name": name,
        "VaR_95": round(var_95, 6),
        "CVaR_95": round(cvar_95, 6),
    })

var_cvar_df = pd.DataFrame(var_rows).sort_values("VaR_95")
var_cvar_df.to_csv(DATA_DIR / "var_cvar_report.csv", index=False)
print(f"Saved: {DATA_DIR / 'var_cvar_report.csv'}")
display(var_cvar_df.head(10))

## Task 2: Rolling 90-Day Sharpe Ratio

The rolling Sharpe tracks how risk-adjusted performance evolves over time using a 90-day window:

$$\text{Rolling Sharpe} = \frac{\text{rolling\_mean}_{90}}{\text{rolling\_std}_{90}} \times \sqrt{252}$$

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for code, label in KEY_FUNDS.items():
    grp = nav_df[nav_df["amfi_code"] == code].sort_values("date")
    rets = grp.set_index("date")["daily_return"]
    roll_mean = rets.rolling(90).mean()
    roll_std = rets.rolling(90).std()
    rolling_sharpe = (roll_mean / roll_std) * np.sqrt(TRADING_DAYS)
    ax.plot(rolling_sharpe.index, rolling_sharpe.values, label=label, linewidth=1.5)

ax.set_title("Rolling 90-Day Sharpe Ratio (5 Key Funds)")
ax.set_xlabel("Date")
ax.set_ylabel("Rolling Sharpe")
ax.legend(loc="best", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CHART_DIR / "rolling_sharpe_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {CHART_DIR / 'rolling_sharpe_chart.png'}")

## Task 3: Investor Cohort Analysis

Investors are grouped by the **year of their first transaction**. For each cohort we compute average SIP amount, total invested amount, and the most preferred fund.

In [ ]:
first_txn = txn_df.groupby("investor_id")["transaction_date"].min().reset_index()
first_txn["cohort_year"] = first_txn["transaction_date"].dt.year
txn_cohort = txn_df.merge(first_txn[["investor_id", "cohort_year"]], on="investor_id")

cohort_rows = []
for year, grp in txn_cohort.groupby("cohort_year"):
    sip_mask = grp["transaction_type"] == "SIP"
    avg_sip = grp.loc[sip_mask, "amount_inr"].mean() if sip_mask.any() else 0
    total_invested = grp.loc[grp["transaction_type"] != "Redemption", "amount_inr"].sum()
    fund_counts = grp.groupby("amfi_code").size()
    top_fund_code = fund_counts.idxmax()
    top_fund = fund_df.loc[fund_df["amfi_code"] == top_fund_code, "scheme_name"].iloc[0]
    cohort_rows.append({
        "cohort_year": year,
        "investors": grp["investor_id"].nunique(),
        "avg_sip_amount_inr": round(avg_sip, 2),
        "total_invested_inr": total_invested,
        "most_preferred_fund": top_fund,
    })

cohort_df = pd.DataFrame(cohort_rows).sort_values("cohort_year")
print("Investor Cohort Summary:")
display(cohort_df)

## Task 4: SIP Continuity Analysis

For investors with **6 or more SIP transactions**, we measure the average gap between consecutive SIP dates. Investors with an average gap **greater than 35 days** are flagged as **at-risk** (likely to discontinue SIP).

In [ ]:
sip_txn = txn_df[txn_df["transaction_type"] == "SIP"].sort_values(
    ["investor_id", "transaction_date"]
)

continuity_rows = []
for inv_id, grp in sip_txn.groupby("investor_id"):
    if len(grp) < 6:
        continue
    gaps = grp["transaction_date"].diff().dt.days.dropna()
    avg_gap = gaps.mean()
    continuity_rows.append({
        "investor_id": inv_id,
        "sip_count": len(grp),
        "avg_gap_days": round(avg_gap, 1),
        "at_risk": avg_gap > 35,
    })

continuity_df = pd.DataFrame(continuity_rows)
eligible = len(continuity_df)
at_risk = continuity_df["at_risk"].sum()
continuity_rate = round((eligible - at_risk) / eligible * 100, 2) if eligible else 0

print("SIP Continuity Summary:")
print(f"  Total eligible investors (6+ SIPs): {eligible}")
print(f"  At-risk investors (avg gap > 35 days): {at_risk}")
print(f"  SIP continuity rate: {continuity_rate}%")
print()
display(continuity_df.head(10))

## Task 5: Simple Fund Recommender

The recommender logic is implemented in `recommender.py`. Run it from the terminal:

```bash
python recommender.py
```

Enter **Low**, **Moderate**, or **High** to get the top 3 funds ranked by Sharpe ratio.

## Task 6: Sector HHI Concentration

The **Herfindahl-Hirschman Index (HHI)** measures portfolio concentration:

$$\text{HHI} = \sum w_i^2$$

where $w_i$ is each holding's weight (%). Higher HHI = more concentrated; lower HHI = more diversified.

In [ ]:
equity_codes = fund_df.loc[fund_df["category"] == "Equity", "amfi_code"].tolist()
equity_holdings = holdings_df[holdings_df["amfi_code"].isin(equity_codes)]

hhi_rows = []
for code, grp in equity_holdings.groupby("amfi_code"):
    weights = grp["weight_pct"] / 100
    hhi = (weights ** 2).sum()
    name = fund_df.loc[fund_df["amfi_code"] == code, "scheme_name"].iloc[0]
    hhi_rows.append({"amfi_code": code, "scheme_name": name, "HHI": round(hhi, 4)})

hhi_df = pd.DataFrame(hhi_rows).sort_values("HHI", ascending=False)
most_concentrated = hhi_df.iloc[0]
most_diversified = hhi_df.iloc[-1]

print(f"Most concentrated: {most_concentrated['scheme_name']} (HHI = {most_concentrated['HHI']})")
print(f"Most diversified:  {most_diversified['scheme_name']} (HHI = {most_diversified['HHI']})")
print()
display(hhi_df.head(10))
print("\nMost diversified funds:")
display(hhi_df.tail(5))

## Task 7: Advanced Insights

### Insight 1: Highest VaR Funds

Funds with the most negative VaR_95 values face the largest single-day losses at the 95% confidence level. Small-cap and mid-cap equity funds typically dominate this list because of higher daily volatility. See the **VaR/CVaR table in Task 1** — the bottom rows show the highest-risk schemes.

### Insight 2: Highest CVaR Funds

CVaR captures the average loss on the worst 5% of trading days — a stricter tail-risk measure than VaR alone. Funds ranking lowest on CVaR_95 experience deeper sustained drawdowns during stress periods. Cross-reference the **var_cvar_report.csv** output; funds where CVaR is substantially below VaR indicate fat left tails in return distribution.

### Insight 3: Investor Cohorts

Grouping investors by first-transaction year reveals how behaviour shifts across vintages. Newer cohorts (2024–2025) tend to show different average SIP amounts and fund preferences compared to earlier entrants. The **cohort table in Task 3** highlights which schemes attract the most investors in each vintage and whether total invested amounts are growing year over year.

### Insight 4: SIP Continuity Findings

Among investors with 6+ SIP transactions, those with average inter-SIP gaps exceeding 35 days are flagged as at-risk of discontinuing. The **continuity summary in Task 4** reports eligible investors, at-risk count, and the overall continuity rate. A lower continuity rate signals a need for investor engagement campaigns to reduce lapse risk.

### Insight 5: Sector Concentration Findings

HHI analysis shows wide variation in portfolio concentration across equity funds. Funds with HHI above 0.15 hold concentrated bets in a few sectors, while diversified large-cap funds sit closer to 0.05–0.10. The **HHI ranking in Task 6** identifies the most concentrated fund (highest HHI) and the most diversified fund (lowest HHI), helping investors match concentration tolerance to fund selection.

## Summary

Day 6 advanced analytics completed:
- **VaR/CVaR** saved to `data/processed/var_cvar_report.csv`
- **Rolling Sharpe chart** saved to `reports/charts/rolling_sharpe_chart.png`
- **Cohort & SIP continuity** analysis displayed above
- **HHI concentration** ranked for all equity funds
- **Fund recommender** available via `recommender.py`